# glr and Go in one notebook

Cells are Go or glr ([golars](https://github.com/Gaurav-Gosain/golars) script): `l` switches a code cell, or start it with `%%glr` / `%%go`. golars frames are shared by name, with no bridge commands:

- a named glr frame (`load PATH as NAME`, `stash NAME`) is a Go variable `NAME` of type `*dataframe.DataFrame`, and the frame a glr cell ends on is the Go variable `glr`;
- a Go variable holding a DataFrame, declared with `:=`, is a glr frame of the same name (`use df`, `join df on id`).

A frame is copied (as Arrow IPC) only when a cell of the other language mentions it and it changed, and every cell ends with a line saying which frames it shared or read.

Needs `golars` for the glr cells and, for the Go cells, the golars module (set `GOLARS_DIR=/path/to/golars` to build against a local checkout).

In [ ]:
# Load with glr. Named frames are visible to Go cells.
load orders.csv as orders
load customers.csv as customers
use orders
head 3

## Transform in Go

`orders` and `customers` are Go variables now. `lines` is declared with `:=`, so it is kept for later cells and shared with glr.

In [ ]:
import (
	"context"

	"github.com/Gaurav-Gosain/golars"
)

// Revenue per order line, with the customer's tier.
lines, err := golars.Lazy(orders).
	WithColumns(golars.Col("unit_price").Mul(golars.Col("qty")).Alias("revenue")).
	Join(golars.Lazy(customers), []string{"customer"}, golars.InnerJoin).
	Collect(context.Background())
if err != nil {
	panic(err)
}
lines.Head(5)

## Summarize in glr

`lines` is a glr frame now.

In [ ]:
use lines
groupby tier revenue:sum:revenue qty:sum:units
sort revenue desc

## And back in Go

The frame the last glr cell ended on is `glr`.

In [ ]:
fmt.Println(glr.Height(), "tiers,", lines.Height(), "order lines")
glr